<a href="https://colab.research.google.com/github/ebmobodo/sentiment-analysis-distilbert/blob/main/Sentiment-analysis-%20distibert" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning DistilBERT for Sentiment Analysis
Author:  Mobodo Ebubechukwu

Goal: In this project, I am going to fine-tune a pre-trained machine learning model called DistilBERT to classify movie reviews as either Positive or Negative.

### 1. Installing Required Libraries
First, I need to install the Hugging Face `transformers` and `datasets` libraries to get my model and data.

In [ ]:
!pip install transformers datasets evaluate accelerate -q

 2. Loading the IMDB Dataset
I am using the IMDB movie review dataset. To save time and compute power, I am shrinking the dataset to 1,000 training examples and 500 testing examples.

In [ ]:
from datasets import load_dataset

# Added the 'stanfordnlp/' namespace to fix the HfUriError
dataset = load_dataset("stanfordnlp/imdb")

# Shrink the dataset for faster training
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

print("Data loaded! Here is a sample review:")
print(small_train[0]['text'])

 3. Tokenizing the Data
AI models can't read English; they only understand numbers. I am using a Tokenizer to convert the movie reviews into a format the DistilBERT model can understand.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)
print("Tokenization complete!")

4. Loading the Pre-Trained Model
I am loading `distilbert-base-uncased`. I am telling it to expect 2 labels: 0 for Negative, and 1 for Positive.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

 5. Setting up Metrics
To know if my model is learning, I need to track its accuracy during training.

In [ ]:
!pip install evaluate -q
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

 6. Fine-Tuning the Model
Now, I will set up the Training Arguments and start the training process. The model will look at the data 3 times (3 epochs).

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification

# Re-defining model to ensure it's available in this cell's scope
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    # THIS IS THE CHANGED LINE:
    output_dir="/content/drive/MyDrive/my_movie_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

 7. Testing the Fine-Tuned Model
Training is done! Now I will write my own custom movie reviews and see if the AI can correctly guess if they are positive or negative.
*(Label 1 = Positive, Label 0 = Negative)*

In [ ]:
from transformers import pipeline

# Set up a pipeline to use our newly trained model
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0)

# Write your own reviews here!
my_reviews = [
    "This was the most amazing and breathtaking movie I have ever seen!",
    "I hated this movie. The acting was terrible and the plot made no sense.",
    "The trailer looked amazing and the actors are incredibly talented, but the actual movie was a complete disaster.",
    "The first half was incredibly boring and I almost walked out. But the twist at the end made it the best movie I've seen all year!"
]

results = classifier(my_reviews)

for review, result in zip(my_reviews, results):
    print(f"Review: {review}")
    print(f"Prediction: {result['label']} (Confidence: {result['score']:.2f})\n")